In [ ]:
import sys
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib as mpl
import numpy as np
import shapely
import spatialdata as sd
import spatialdata_plot
import zarr
from anndata.io import read_elem
from matplotlib import pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.patches import Patch
from shapely import affinity
from spatialdata.models import Image2DModel, ShapesModel
from spatialdata.transformations import Affine, Scale, Sequence, get_transformation, set_transformation

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import figstyle as fs

warnings.filterwarnings("ignore", message="SpatialData is not stored in the most current format")

In [ ]:
XENIUM_ZARR = Path("..").resolve() / "xenium_data_mouse" / "integrated" / "integrated_mouse.zarr"
ANNOTATED_ZARR = Path("data") / "integrated_mouse_annotated.zarr"
OUT_DIR = Path("results") / "02_fibroblast_mapping"

FRAMES = {
    "γ-ray Sham (ROI 8)": ((4310, 16140), 5, 0),
    "C-ion 3.5 d (ROI 6)": ((1810, 15440), 30, 90),
    "γ-ray 3.5 d (ROI 13)": ((7800, 14400), 350, 0),
}
FRAME_SIZE = 700
SCALE_BAR = 50
GRID_WIDTH = 180 * fs.MM

CHANNEL_COLORS = {"DAPI": "#3D7BFF", "ATP1A1/CD45/E-Cadherin": "#FF00FF", "alphaSMA/Vimentin": "#00FF00"}
PERCENTILE = (1, 99)
ZOOM_BRIGHTNESS = 0.5
DOWNSAMPLE = 8

CELLTYPE_KEY = "cell_subtype"
CELL_TYPES = ["Telocyte", "Trophocyte", "Adamdec1+ fibroblast", "Villus fibroblast"]

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

obs = read_elem(zarr.open_group(ANNOTATED_ZARR, mode="r")["obs"])[["section", "condition", "slide", CELLTYPE_KEY]]
sdata = sd.read_zarr(XENIUM_ZARR, selection=("images", "shapes"))

def rotation(center, deg):
    (cx, cy), t = center, np.deg2rad(deg)
    c, s = np.cos(t), np.sin(t)
    return Affine([[c, -s, cx - c * cx + s * cy], [s, c, cy - s * cx - c * cy], [0, 0, 1]],
                  input_axes=("x", "y"), output_axes=("x", "y"))

def frame_polygon(center, size, angle):
    half = size / 2
    return affinity.rotate(shapely.box(center[0] - half, center[1] - half, center[0] + half, center[1] + half), -angle, origin=center)

sections = {}
for section, (center, angle, turn) in FRAMES.items():
    obs_sec = obs[obs["section"] == section]
    roi_id = f"{obs_sec['condition'].iloc[0]}_{obs_sec['slide'].iloc[0]}"
    image = sdata[f"morphology_focus_{roi_id}"]

    image_small = image.coarsen(y=DOWNSAMPLE, x=DOWNSAMPLE, boundary="trim").mean().compute()
    image_small = Image2DModel.parse(image_small.values, dims=("c", "y", "x"), c_coords=image.c.values, transformations={
        "global": Sequence([Scale([DOWNSAMPLE, DOWNSAMPLE], axes=("x", "y")), get_transformation(image, "global")])})

    obs_shown = obs_sec[obs_sec[CELLTYPE_KEY].isin(CELL_TYPES)]
    boundaries = ShapesModel.parse(
        gpd.GeoDataFrame(obs_shown[[CELLTYPE_KEY]], geometry=sdata["cell_boundaries"].geometry.loc[obs_shown.index].values),
        transformations={"global": get_transformation(sdata["cell_boundaries"], "global")})

    sec = sections[section] = sd.SpatialData(
        images={"image": image, "image_small": image_small},
        shapes={"boundaries": boundaries, "frame": ShapesModel.parse(gpd.GeoDataFrame(geometry=[frame_polygon(center, FRAME_SIZE, angle)]))})
    for name in ("image", "boundaries"):
        set_transformation(sec[name], Sequence([get_transformation(sec[name], "global"), rotation(center, angle - turn)]), "zoom")
    print(section, len(obs_sec), "cells")

In [ ]:
limits = {}
for channel in CHANNEL_COLORS:
    pixels = np.concatenate([sec["image_small"].sel(c=channel).values.ravel() for sec in sections.values()])
    limits[channel] = np.percentile(pixels[pixels > 0], PERCENTILE)

def render_image(sec, name, alpha=1):
    return sec.pl.render_images(name, channel=list(CHANNEL_COLORS), palette=list(CHANNEL_COLORS.values()),
                                norm=[Normalize(*limits[channel]) for channel in CHANNEL_COLORS], scale="full", alpha=alpha)

def finish_image(ax):
    ax.images[0].set_interpolation("none")
    ax.set_facecolor("black")

In [ ]:
for section, sec in sections.items():
    fig, ax = plt.subplots(figsize=(fs.COL1, fs.COL1), layout="constrained")
    (render_image(sec, "image_small")
     .pl.render_shapes("frame", fill_alpha=0, outline_alpha=1, outline_color=fs.YELLOW, outline_width=0.75)
     .pl.show(coordinate_systems="global", ax=ax, legend_loc=None, colorbar=False))
    finish_image(ax)

    extent = sd.get_extent(sec["image_small"])
    ax.set(xlim=extent["x"], ylim=extent["y"][::-1], aspect="equal", xlabel="x (µm)", ylabel="y (µm)", title=section)
    ax.xaxis.set_major_locator(mpl.ticker.MultipleLocator(500))
    ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(500))
    ax.tick_params(axis="x", labelrotation=90)
    ax.grid(color="white", lw=0.25, alpha=0.5)
    plt.show()

In [ ]:
def scale_bar(ax, length, loc=(0.05, 0.05), color="black"):
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    x = min(x0, x1) + loc[0] * abs(x1 - x0)
    y = y0 + loc[1] * (y1 - y0)
    ax.plot([x, x + length], [y, y], color=color, lw=1.0, solid_capstyle="butt")
    ax.annotate(f"{length:g} µm", (x + length / 2, y), xytext=(0, 2), textcoords="offset points",
                ha="center", va="bottom", fontsize=6, color=color)

celltype_colors = dict(zip(CELL_TYPES, fs.palette(CELL_TYPES)))

fig, axes = plt.subplots(1, len(FRAMES), figsize=(GRID_WIDTH, GRID_WIDTH / len(FRAMES) + 15 * fs.MM), layout="constrained")
for ax, (section, sec) in zip(axes, sections.items()):
    (cx, cy), angle, turn = FRAMES[section]
    half = FRAME_SIZE / 2

    cropped = sec.filter_by_coordinate_system("zoom").query.bounding_box(
        axes=["x", "y"], min_coordinate=[cx - half, cy - half], max_coordinate=[cx + half, cy + half], target_coordinate_system="zoom")

    (render_image(cropped, "image", alpha=ZOOM_BRIGHTNESS)
     .pl.render_shapes("boundaries", color=CELLTYPE_KEY, palette=celltype_colors, fill_alpha=1,
                       outline_alpha=1, outline_color="black", outline_width=0.1)
     .pl.show(coordinate_systems="zoom", ax=ax, legend_loc=None, colorbar=False))
    finish_image(ax)

    ax.set(xlim=(cx - half, cx + half), ylim=(cy + half, cy - half), aspect="equal", xticks=[], yticks=[], ylabel="", title="")
    ax.set_xlabel(section, labelpad=3)
    ax.spines[:].set_visible(False)
    ax.apply_aspect()
    scale_bar(ax, SCALE_BAR, loc=(0.05, 0.04), color="white")

fig.legend(handles=[Patch(fc=color, ec="black", lw=0.3, label=celltype) for celltype, color in celltype_colors.items()],
           loc="outside lower center", ncol=len(CELL_TYPES), handlelength=1, handleheight=0.8)
fs.save(fig, OUT_DIR / f"zoom_{FRAME_SIZE}um")
plt.show()